# Colab Experiment: Unlearnable Transfer Victims

Goal: reuse the protected dataset generated by the full Unlearnable run and train additional victim architectures to test transferability. This notebook does not regenerate PGD noise.


## 1. Setup Colab / GitHub repo


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
PROJECT_DIR = "adversarial-data-protection"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src").exists():
    if not Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 2. Mount Google Drive


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"
DRIVE_DATA_ROOT = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_RESULTS_DIR = f"{DRIVE_PROJECT_DIR}/results"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = DRIVE_DATA_ROOT
        RESULTS_ROOT = DRIVE_RESULTS_DIR
        Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
        Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
    except ImportError:
        print("Not running in Colab; using local ./data and ./results")

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


## 3. Train Additional Victim Architectures

This cell loads `protected_dataset.pt`, trains clean/protected versions of each victim architecture, and evaluates both on the clean CIFAR-10 test set.


In [ ]:
import os
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm

from src.models import evaluate, get_victim_mobilenet, get_victim_vgg16, train_one_epoch

SEED = 42
EPSILON = 0.03
EPOCHS = 15
LEARNING_RATE = 0.05
WEIGHT_DECAY = 5e-4
TEST_SUBSET_SIZE = 10000

# Reuse the protected dataset produced by the full-final ResNet-18 run.
SOURCE_RUN_DIR = Path(RESULTS_ROOT) / "unlearnable_full_final" / "train45000_val5000_seed42_base20_victim20_pgd10_inner2"
PROTECTED_DATASET_PATH = SOURCE_RUN_DIR / "tensors" / "eps0p03" / "protected_dataset.pt"

RUN_NAME = f"transfer_victims_eps{EPSILON}_seed{SEED}_epochs{EPOCHS}_from_full_final"
RUN_DIR = Path(RESULTS_ROOT) / "unlearnable_transfer_victims" / RUN_NAME
TABLE_DIR = RUN_DIR / "tables"
MODEL_DIR = RUN_DIR / "models"
for path in [TABLE_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)


class CifarNormalize(nn.Module):
    def __init__(self, mean=CIFAR10_MEAN, std=CIFAR10_STD):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean.to(x.device, x.dtype)) / self.std.to(x.device, x.dtype)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def format_seconds(seconds):
    seconds = int(round(seconds))
    minutes, sec = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {sec}s"
    if minutes:
        return f"{minutes}m {sec}s"
    return f"{sec}s"


def wrap_model(model_fn, device):
    return nn.Sequential(CifarNormalize(), model_fn(num_classes=10, device="cpu")).to(device)


def make_loader(dataset, batch_size, shuffle, seed_offset=0):
    generator = torch.Generator().manual_seed(SEED + seed_offset)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=True,
        generator=generator if shuffle else None,
    )


def train_classifier(model_fn, loader, epochs, device, lr=LEARNING_RATE, desc="Training"):
    model = model_fn().to(device)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
        nesterov=True,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    history = []
    progress = tqdm(range(epochs), desc=desc, leave=True)
    start_all = time.perf_counter()
    for epoch in progress:
        start_epoch = time.perf_counter()
        loss, acc = train_one_epoch(model, loader, optimizer, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        epoch_seconds = time.perf_counter() - start_epoch
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": loss,
                "train_accuracy": acc,
                "lr": round(current_lr, 6),
                "epoch_seconds": round(epoch_seconds, 2),
            }
        )
        progress.set_postfix(loss=loss, acc=acc, lr=f"{current_lr:.5f}", time=format_seconds(epoch_seconds))
        print(f"  epoch {epoch + 1}/{epochs} loss={loss} train_acc={acc} lr={current_lr:.6f} time={format_seconds(epoch_seconds)}")
    total_seconds = time.perf_counter() - start_all
    print(f"{desc} completed in {format_seconds(total_seconds)}")
    return model, history, total_seconds


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("protected dataset:", PROTECTED_DATASET_PATH)
print("RUN_DIR:", RUN_DIR)

if not PROTECTED_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Protected dataset not found: {PROTECTED_DATASET_PATH}. Run colab_unlearnable_full_final_experiment.ipynb first."
    )

checkpoint = torch.load(PROTECTED_DATASET_PATH, map_location="cpu")
x_clean = checkpoint["x_clean"].float().clamp(0, 1)
x_protected = checkpoint["x_protected"].float().clamp(0, 1)
y = checkpoint["y"].long()
print("loaded tensors:", x_clean.shape, x_protected.shape, y.shape)

clean_train_dataset = TensorDataset(x_clean, y)
protected_train_dataset = TensorDataset(x_protected, y)

test_transform = T.Compose([T.ToTensor()])
test_full = CIFAR10(root=DATA_ROOT, train=False, download=True, transform=test_transform)
test_indices = list(range(min(TEST_SUBSET_SIZE, len(test_full))))
test_dataset = torch.utils.data.Subset(test_full, test_indices)
test_loader = make_loader(test_dataset, batch_size=256, shuffle=False)
print("test size:", len(test_dataset))

VICTIM_CONFIGS = {
    "MobileNetV2": {
        "fn": get_victim_mobilenet,
        "batch_size": 128,
        "lr": 0.05,
    },
    "VGG-16": {
        "fn": get_victim_vgg16,
        "batch_size": 64,
        "lr": 0.01,
    },
}

records = []
total_start = time.perf_counter()
for model_name, cfg in VICTIM_CONFIGS.items():
    print("\n" + "=" * 80)
    print("Transfer victim:", model_name)
    model_dir = MODEL_DIR / model_name.replace("/", "_").replace(" ", "_")
    model_dir.mkdir(parents=True, exist_ok=True)

    clean_loader = make_loader(clean_train_dataset, batch_size=cfg["batch_size"], shuffle=True, seed_offset=10)
    protected_loader = make_loader(protected_train_dataset, batch_size=cfg["batch_size"], shuffle=True, seed_offset=20)

    set_seed(SEED)
    baseline_model, baseline_history, baseline_seconds = train_classifier(
        lambda fn=cfg["fn"]: wrap_model(fn, device),
        clean_loader,
        EPOCHS,
        device,
        lr=cfg["lr"],
        desc=f"{model_name} clean baseline",
    )
    baseline_test_acc = evaluate(baseline_model, test_loader, device)
    pd.DataFrame(baseline_history).to_csv(TABLE_DIR / f"{model_name}_baseline_history.csv", index=False)
    torch.save(
        {
            "model_state_dict": baseline_model.state_dict(),
            "model": model_name,
            "train_data": "clean_full_final_train_split",
            "epochs": EPOCHS,
            "seed": SEED,
            "test_accuracy": baseline_test_acc,
        },
        model_dir / "baseline_clean_model.pt",
    )
    del baseline_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    set_seed(SEED)
    protected_model, protected_history, protected_seconds = train_classifier(
        lambda fn=cfg["fn"]: wrap_model(fn, device),
        protected_loader,
        EPOCHS,
        device,
        lr=cfg["lr"],
        desc=f"{model_name} protected victim",
    )
    protected_test_acc = evaluate(protected_model, test_loader, device)
    asr_proxy = round(1.0 - protected_test_acc, 4)
    acc_drop = round(baseline_test_acc - protected_test_acc, 4)
    pd.DataFrame(protected_history).to_csv(TABLE_DIR / f"{model_name}_protected_history.csv", index=False)
    torch.save(
        {
            "model_state_dict": protected_model.state_dict(),
            "model": model_name,
            "train_data": "protected_full_final_train_split",
            "epochs": EPOCHS,
            "epsilon": EPSILON,
            "seed": SEED,
            "test_accuracy": protected_test_acc,
        },
        model_dir / "victim_protected_model.pt",
    )
    del protected_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    record = {
        "technique": "unlearnable_transfer_victim",
        "surrogate_model": "ResNet-18+CIFAR-normalization",
        "victim_model": model_name,
        "epsilon": EPSILON,
        "seed": SEED,
        "epochs": EPOCHS,
        "train_size": len(clean_train_dataset),
        "test_size": len(test_dataset),
        "baseline_clean_test_accuracy": baseline_test_acc,
        "protected_clean_test_accuracy": protected_test_acc,
        "accuracy_drop": acc_drop,
        "asr_proxy": asr_proxy,
        "baseline_training_seconds": round(baseline_seconds, 2),
        "protected_training_seconds": round(protected_seconds, 2),
        "source_protected_dataset": str(PROTECTED_DATASET_PATH),
        "run_dir": str(RUN_DIR),
    }
    print(record)
    records.append(record)
    pd.DataFrame(records).to_csv(TABLE_DIR / "transfer_victim_results_partial.csv", index=False)

results_df = pd.DataFrame(records)
results_df.to_csv(TABLE_DIR / "transfer_victim_results.csv", index=False)
print("\nSaved:", TABLE_DIR / "transfer_victim_results.csv")
print("Total runtime:", format_seconds(time.perf_counter() - total_start))
results_df
